## Legal Expert Knowledge Worker

This is a question-answering assistant with access to the Constitution (supreme law), statutory law, English common law, customary law and Islamic law documents. This will make it an expert in legal problems and it will answer any legal questions thrown to it.

### Note: The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

### PART A: Retrive the documents and Divide them into chunks

The documents will be retrived from an external source. e.g. Google Drive

In [1]:
%pip install google-api-python-client google-auth google-auth-oauthlib
%pip install 'markitdown[all]'


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

import tiktoken
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

from google_drive_client import GoogleDriveClient

google_drive_credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
google_drive_folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')

print(f"Credentials path: {google_drive_credentials_path}")
print(f"Google Drive Folder ID: {google_drive_folder_id}")

# Google Drive client:
collector = GoogleDriveClient(google_drive_credentials_path, google_drive_folder_id)
collector.collect_files()

# Files from a Google Drive folder in markdown format.
documents = collector.get_collection()

# Entire knowledge base
entire_knowledge_base = collector.entire_knowledge_base

print(f"Found {len(documents)} files in the knowledge base")

/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Credentials path: /home/thomas/Downloads/googleUserContent.json
Google Drive Folder ID: 1dOFomY-wQD0NmBDA1XAH8eUqLaE2A0iz
Total characters in knowledge base: 3,336,279
Found 2 files in the knowledge base


In [2]:

from openai import OpenAI

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

Check the number of tokens.

In [ ]:
# How many tokens in all the documents?
model = MODEL_MAP["GPT"]["model"]

def count_tokens(model, knowledge_base):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(entire_knowledge_base)
    token_count = len(tokens)
    return token_count

token_count = count_tokens(model, entire_knowledge_base)

print(f"Total tokens for {model}: {token_count:,}")

# print(documents['The_Constitution_of_Kenya_2010.md'][0:1000])
print(documents['companies-act-2015.md'][0:1000])


Total tokens for gpt-4o-mini: 827,763
SPECIAL ISSUE
Kenya Gazette Supplement No. 158 (Acts No. 17)
REPUBLIC OF KENYA
–––––––
KENYA GAZETTE SUPPLEMENT
ACTS, 2015
NAIROBI, 15th September, 2015
CONTENT
Act—
PAGE
The Companies Act, 2015.............................................................................267
PRINTED AND PUBLISHED BY THE GOVERNMENT PRINTER, NAIROBI

267
THE COMPANIES ACT
No. 17 of 2015
Date of Assent: 11th September, 2015
Date of Commencement: Section 2 on 15th September, 2015
All other provisions: See Section 1 (3) and (4)
ARRANGEMENT OF SECTIONS
Section
PART I—PRELIMINARY
1—Short title and commencement.
2—Objects of this Act.
3—Interpretation of provisions of this Act.
4—Provisions supplementing definition of “holding
company” in section 3.
PART II—COMPANIES AND COMPANY
FORMATION
Division 1—Types of companies
5—Limited companies.
6—Companies limited by shares.
7—Companies limited by guarantee.
8—Unlimited companies.
9—Private companies.
10—Public companies.
Divisio

Divide the documents into chunks.

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter
# But first convert the documents to a list of langchain documents.
from langchain_core.documents import Document

langchain_documents = [
    Document(page_content=content, metadata={"source": filename})
    for filename, content in documents.items()
]

def split_documents(langchain_documents, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(langchain_documents)
    return chunks

chunks = split_documents(langchain_documents, 1000, 200)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

chunks[100]

Divided into 4359 chunks
First chunk:

page_content='SPECIAL ISSUE
Kenya Gazette Supplement No. 158 (Acts No. 17)
REPUBLIC OF KENYA
–––––––
KENYA GAZETTE SUPPLEMENT
ACTS, 2015
NAIROBI, 15th September, 2015
CONTENT
Act—
PAGE
The Companies Act, 2015.............................................................................267
PRINTED AND PUBLISHED BY THE GOVERNMENT PRINTER, NAIROBI' metadata={'source': 'companies-act-2015.md'}


Document(metadata={'source': 'companies-act-2015.md'}, page_content='| ---------------- | --- | --- | --- | --------- | --- | ---- | -------- | ---------- | --- |\nstatements with Registrar.\n686—Lodgement requirements for companies subject to\nsmall companies regime.\n687—Lodgement requirements for unquoted companies.\n688—Lodgement requirements for quoted companies.\n| 689—Exemption  |              |     | of  | unlimited  |            |     | companies   | from  |     |\n| -------------- | ------------ | --- | --- | ---------- | ---------- | --- | ----------- | ----- | --- |\n|                | requirement  |     | to  | lodge      | financial  |     | statements  | with  |     |\nRegistrar.\n| 690—Special  |     | auditor´s  |     | report  |     | required  | if  | abbreviated  |     |\n| ------------ | --- | ---------- | --- | ------- | --- | --------- | --- | ------------ | --- |\nfinancial statement is lodged with Registrar.\n| 691—Directors  |     |     | of  | company  |     

### PART B: Make vectors and store in Chroma

In [ ]:
# Pick an embedding model
db_name = "vector_db"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 4359 documents


In [7]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 4,359 vectors with 384 dimensions in the vector store
